[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/mlengineering/blob/main/advanced/notebooks/03_package_structure.ipynb)

*Click the badge to run this notebook in Google Colab - no install required. The first code cell sets everything up.*

# 03 · From script to installable package: project structure and environments

**Week 1 · day 3 · 2 h essential path + up to 1 h optional** · lab files: [`labs/w1d3_package_structure/`](../../labs/w1d3_package_structure/)

**Objective.** Turn a notebook-export script into a src-layout package with pyproject.toml, an isolated environment, an editable install and a console entry point, proving behaviour is preserved.

In [ ]:
# --- Setup: works on your laptop AND in Google Colab ----------------------------
# In Colab this clones the course repository and installs the requirements (about a
# minute, first run only). On your own machine it just finds the repository. Every
# command in this notebook runs from the repository root. Safe to re-run.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/mlengineering").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/berdakh/mlengineering.git",
                        "/content/mlengineering"], check=True)
    os.chdir("/content/mlengineering")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src" / "mlcourse").is_dir())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print("repo root:", REPO_ROOT)

**Unit id:** `w1d3` · **Week 1, core day 3** · **Handbook:** [H2.1 Project structure and environments](../../docs/handbook/ch02_maintainable_python/h2_1_project_structure_environments.md)
**Data:** SYNTHETIC pump vibration (`data/synthetic/pump_vibration.csv`, schema in `data/README.md`)

> Commands are run from the repository root unless a step says otherwise.

---

## 1. Purpose

**Primary objective.** Turn a notebook-export script into an installable `src`-layout
package with a `pyproject.toml`, an isolated environment, an editable install and a
console command — and *prove* with a characterization run that the refactor preserved
behaviour before making one deliberate, recorded behaviour change.

**Why an applied ML engineer needs it.** Everything later in this course — tests, an
inference API, a Docker image, CI — consumes your code as an *installed package with
declared dependencies*, not as a file that happens to run in the directory where you
wrote it. "Works on my machine, from this folder, in this conda env" is the most common
reason research code cannot be handed to anyone. The second skill here is less visible
and more valuable: changing structure without changing results, and being able to show
that you did.

**You will learn to**

- explain where `import pumpwatch` looks, and why the answer depends on how Python was started;
- lay out a project so the *installed* code is what runs and is tested;
- write a minimal, correct `pyproject.toml` (build system, metadata, dependencies, entry point);
- create and use a virtual environment without relying on activation magic;
- separate behaviour-preserving changes from behaviour-changing ones and verify each appropriately.

**Prerequisites.** Fluent Python; you have used `pip` and some environment manager.
Diagnostic sections on Python refactoring (w1d1) attempted. No packaging knowledge assumed.

**Deliverable.** The `lab/` directory containing an installable `pumpwatch` package, `golden.txt`,
and a filled-in `REFACTOR_LOG.md`.

**Completion criteria.**

- [ ] `python -m pytest labs/w1d3_package_structure/checks -q` passes (guided practice).
- [ ] `pumpwatch train --data <absolute path>` prints the golden metrics when run from `/tmp`.
- [ ] `REFACTOR_LOG.md` lists at least four issues in the original script and separates
      behaviour-preserving from behaviour-changing steps, with before/after numbers.
- [ ] You answered the assessment questions *in writing* before opening the answers.
- [ ] (Independent skill) the challenge checks pass without opening hint 3 or the solution.

**Time budget — essential path, 2 h**

| Activity | Minutes |
|---|---|
| Sections 2–3: concepts and worked example (run it) | 35 |
| Section 4: guided lab | 65 |
| Section 7: assessment questions, written | 10 |
| Section 8: reflection, mark progress | 10 |

**Optional extension, ≤ 1 h:** independent challenge (≈ 45 min) and the interview rep (≈ 15 min).
If you stop at two hours, do the challenge on the review day; "independent skill" stays
unmarked until then.

---

## 2. Conceptual explanation

The full treatment, with more examples and the tradeoff tables, is handbook
[H2.1](../../docs/handbook/ch02_maintainable_python/h2_1_project_structure_environments.md).
This section is the minimum you need for the lab.

### 2.1 Intuition: an import is a search, and you control the search path

`import pumpwatch` walks the list `sys.path` in order and takes the first match. The list
is built at interpreter start-up:

1. **Position 0 depends on how you launched Python.** `python path/to/script.py` puts the
   *script's directory* first. `python -m module`, `python -c ...` and the REPL put the
   *current working directory* first.
2. Then anything in the `PYTHONPATH` environment variable.
3. Then the standard library.
4. Finally the environment's `site-packages` — where `pip install` puts things.

Consequences you have certainly met:

- a notebook imports `utils.py` because Jupyter's working directory happens to contain it;
  move the notebook and it breaks;
- `sys.path.append("../src")` at the top of a script — a search-path patch that is only
  right for one working directory;
- a file named `random.py` or `signal.py` next to your script shadows the standard library.

*Installing* a package means making it findable through step 4, which is the only step
that does not depend on where you are standing or how you started Python.

<img src="https://raw.githubusercontent.com/berdakh/mlengineering/main/docs/assets/diagrams/mmd-03-01.svg" alt="Diagram 1 of notebook 03" style="max-width:100%">

*An import is a search over `sys.path`, in order. "Works from the repo root, fails elsewhere" means entry 0 found your code and nothing else would have.*

### 2.2 Precise terminology

| Term | Meaning |
|---|---|
| **module** | one importable `.py` file |
| **import package** | a directory of modules you `import` (`pumpwatch/`) |
| **distribution package** (project) | the installable, versioned thing pip deals with (`pumpwatch 0.1.0`). Its name can differ from the import name (`scikit-learn` → `sklearn`). |
| **sdist / wheel** | source archive / built archive that pip installs by unpacking |
| **build frontend** | the tool you invoke: `pip`, `python -m build`, `uv` |
| **build backend** | the library that turns your source tree into a wheel: `setuptools`, `hatchling`, … Named in `[build-system]`; the frontend downloads it into a temporary build environment. |
| **virtual environment** | a directory with its own `site-packages` and a `pyvenv.cfg` pointing at a base interpreter. Cheap, disposable, not relocatable. |
| **editable install** | an install whose `site-packages` entry *points back at your source tree*, so edits take effect without reinstalling |
| **entry point / console script** | `name = "module:callable"` in `[project.scripts]`; the installer generates an executable `name` that imports the callable and calls it |

### 2.3 Why the `src` layout

Two layouts:

```
flat                              src
project/                          project/
  pyproject.toml                    pyproject.toml
  pumpwatch/__init__.py             src/pumpwatch/__init__.py
```

With the flat layout, standing in `project/` and running `python -m pytest` or
`python -c "import pumpwatch"` puts `project/` at `sys.path[0]`, so `pumpwatch` is
imported **from the working tree whether or not it is installed** — and whether or not
the installed copy is complete. You can ship a wheel that is missing a subpackage or a
data file while every local test passes.

With `src/`, the project root contains no importable `pumpwatch`. The only way to import
it is through an installation. If the packaging configuration is wrong, you find out
immediately, on your machine. The cost is one extra step (`pip install -e .`) before
anything runs — which is the step your colleague and your Dockerfile need anyway.

```
flat layout                              src layout
-----------                              ----------
project/                                 project/
  pumpwatch/        <- importable from     pyproject.toml
    __init__.py        the repo root       src/
  pyproject.toml       WITHOUT installing    pumpwatch/      <- importable ONLY
  tests/                                       __init__.py      after installing
                                           tests/

python -c "import pumpwatch"             python -c "import pumpwatch"
  from project/      -> works (entry 0)    from project/      -> fails until installed
  from elsewhere     -> fails              from anywhere      -> works once installed
  => a broken or missing install           => tests always run against the
     can go unnoticed                         installed package
```

*The `src` layout removes the accidental import path, so "it imports" means "it is installed".*

### 2.4 What `pip install -e .` actually does

1. pip reads `[build-system]`, creates an isolated temporary environment, and installs
   the backend (`setuptools`) into it.
2. The backend reads `[project]` and writes metadata into `site-packages`
   (`pumpwatch-0.1.0.dist-info/`): name, version, dependencies, entry points.
3. Instead of copying your code, it writes a small pointer (for setuptools, a `.pth` file
   or an import hook) that makes `src/` importable.
4. pip installs the declared `dependencies`.
5. For each `[project.scripts]` entry it writes an executable into the environment's `bin/`.

Edits to existing `.py` files are live. **Metadata is not live**: after changing
dependencies or entry points in `pyproject.toml`, re-run `pip install -e .`. An editable
install is a development convenience, not a deployment mechanism — images and servers get
a regular install (week 6).

<img src="https://raw.githubusercontent.com/berdakh/mlengineering/main/docs/assets/diagrams/mmd-03-02.svg" alt="Diagram 2 of notebook 03" style="max-width:100%">

*What an editable install does. The frontend orchestrates; the backend knows how to build your project.*

### 2.5 Environments: isolation, then reproducibility

A virtual environment answers *"which packages can this interpreter see?"* Run
`.venv/bin/python` and you are in the environment; `source .venv/bin/activate` merely puts
`.venv/bin` first on `PATH` so that `python` and `pip` resolve there. Always prefer
`python -m pip` over bare `pip`: it guarantees you install into the interpreter you are
running.

Two different files describe dependencies, and conflating them causes real damage:

- **Abstract requirements** (`dependencies` in `pyproject.toml`): what the package needs,
  with the *loosest* bounds you believe (`numpy>=1.26`). Read by anyone who installs it.
- **A lock / pinned set** (`requirements.txt` from `pip freeze`, `uv.lock`, …): the *exact*
  versions of everything, including transitive dependencies, that one environment
  resolved to. Used to rebuild *that application environment*.

A library pins loosely so it can coexist with others; a deployed service locks tightly so
Tuesday's build equals Monday's. Your project will be both: a package (loose) and, from
week 6, a service image (locked).

### 2.6 Connection to other domains

- The feature set is standard signal processing: **line length** `Σ|x[t+1]−x[t]|` (a
  classic feature in EEG seizure detection), **RMS**, and **band power** from a periodogram.
  A bearing fault adds narrow-band energy near a characteristic frequency, much as an
  oscillation in a biomedical signal adds energy in its band; `ratio = bp_high / bp_low` is
  the familiar relative-band-power trick, cancelling the per-unit sensor gain.
- `unit_id` plays the role of a *subject* in a medical study or a *machine* in a fleet. The
  script already uses a grouped split, but look at where it fits the scaler.
- A characterization run is a pilot recording: before changing the rig, record what the
  current rig produces, so later differences can be attributed.

<img src="https://raw.githubusercontent.com/berdakh/mlengineering/main/labs/w1d3_package_structure/figures/pump_signal_spectrum.svg" alt="SYNTHETIC pump data: a healthy and a faulty window of unit P09 in time and frequency; the faulty window has an extra peak at 90 Hz inside the bp_high band" width="100%">

*SYNTHETIC data. The features you will move into `pumpwatch.features` are standard window features from signal processing: here `bp_high` picks up the 90 Hz fault component and `bp_low` the shaft rotation near 25 Hz. Regenerate with `python labs/w1d3_package_structure/figures/make_figures.py`.*

<img src="https://raw.githubusercontent.com/berdakh/mlengineering/main/labs/w1d3_package_structure/figures/unit_vs_fault.svg" alt="SYNTHETIC pump data: per-unit strip plots; mean rms per unit ranges from 0.36 to 1.27 while healthy and faulty windows overlap, and the bp_high/bp_low ratio separates the classes within every faulty unit" width="100%">

*SYNTHETIC data. Left: mean `rms` per unit runs from 0.36 to 1.27, a wider spread than the healthy-versus-faulty gap (0.81 versus 1.01 on average), which is why the split is by unit, as it would be by subject in a medical study. Right: a ratio of band powers cancels the unit gain.*

### 2.7 A concrete industry example

A data scientist hands over `churn_model_final.ipynb` plus a pickle. The platform team
must run scoring nightly. The notebook imports `helpers` from the author's home
directory, depends on whatever was in their conda env in March, and reads
`../data/latest.csv`. Three weeks go to archaeology. The replacement is unglamorous: a
repository with `pyproject.toml`, `src/churn/`, a `churn score --input … --output …`
command, and a pinned environment. The nightly job becomes two lines —
`pip install churn==1.4.2` and one command — and when a number looks wrong, "which code
produced this?" has an answer: a version.

### 2.8 Assumptions, limits, and common misconceptions

- **"A package is for publishing on PyPI."** No. You install your own project locally;
  nothing leaves your machine. Packaging is about a stable import path and declared
  dependencies.
- **"`__init__.py` makes it a package, so I'm done."** It makes the directory importable
  *if it is on `sys.path`*. Installation is what puts it there reliably.
- **"Editable means everything updates."** Code yes; metadata, new entry points and
  dependencies no.
- **"`pip freeze > requirements.txt` is my dependency declaration."** It is a snapshot of
  one environment, including things you installed by accident. Useful as a lock, harmful
  as the *only* description.
- **"conda/uv/poetry make this obsolete."** They are frontends and environment managers
  around the same standards (`pyproject.toml`, wheels, `site-packages`). The model above
  is what they all implement.
- **"If the metrics are identical after my change, the change was harmless / the bug was
  not real."** You will see a counterexample in Task 7.
- **Limit of this lesson:** a correct package does not make the *analysis* correct.
  Single split, three test units, a fixed 0.5 threshold — all still there. Those are
  weeks 2–3.

---

## 3. Worked example: `room_report.py` → `tempstats`

A deliberately tiny, different problem so the lab is not a copy exercise.
Files: [`example/`](../../labs/w1d3_package_structure/example).

**Dataset (SYNTHETIC).** `example/data/room_temps.csv`, 72 rows, hourly readings for
three rooms over one day, two missing values.

| column | type | meaning |
|---|---|---|
| `room` | str | `lab_a`, `lab_b`, `server` |
| `timestamp` | ISO-8601 str | hourly, 2026-03-02 |
| `temp_c` | float, nullable | temperature in °C |

**Before** — [`example/before/room_report.py`](../../labs/w1d3_package_structure/example/before/room_report.py):

In [ ]:
!python labs/w1d3_package_structure/example/before/room_report.py

Expected output (exact; the data file is committed):

```
lab_a 21.01 4.28 23
lab_b 23.48 3.31 23
server 26.92 3.23 24
```

Now run it from somewhere else — `cd /tmp && python /path/to/.../before/room_report.py` —
and it fails with `FileNotFoundError`: the data path is relative to the *working
directory*, not to the script.

**After** — [`example/after/`](../../labs/w1d3_package_structure/example/after): `pyproject.toml`, `src/tempstats/stats.py`,
`src/tempstats/cli.py`.

```bash
python -m venv /tmp/tempstats-venv
/tmp/tempstats-venv/bin/python -m pip install -e labs/w1d3_package_structure/example/after
cd /tmp
/tmp/tempstats-venv/bin/tempstats /absolute/path/to/labs/w1d3_package_structure/example/data/room_temps.csv
/tmp/tempstats-venv/bin/python -c "import tempstats; print(tempstats.__file__)"
```

Expected: the same three lines as before, from any directory; the last command prints a
path inside `example/after/src/tempstats/` (the editable pointer at work). No network?
`PYTHONPATH=labs/w1d3_package_structure/example/after/src python -m tempstats.cli <csv>`
exercises the same code without installing — useful for a quick look, but note that it is
precisely the search-path patching the install replaces.

**Code decisions worth noticing**

1. **`stats.summarize(df) -> df` knows nothing about files or printing.** The I/O boundary
   is `cli.main`. This is the split that makes the function testable (w1d5) and reusable
   behind an API (week 5).
2. **`main(argv=None) -> int` and `sys.exit(main())`.** Returning the exit code instead of
   calling `sys.exit` inside makes `main(["file.csv"])` callable from a test. The generated
   console script does exactly `sys.exit(main())`.
3. **Validation at the boundary.** `summarize` raises `ValueError` naming missing columns;
   `main` converts expected failures into a one-line message and exit code 2. Unexpected
   exceptions still produce a traceback — that is correct; do not catch `Exception`.
4. **`groupby(..., sort=False)`** preserves the script's first-seen room order. Without
   it the rows would come out sorted — the same here, different for other data. Small
   behaviour differences like this are what a characterization run catches.
5. **`requires-python = ">=3.11"`, `dependencies = ["pandas>=2.1"]`.** Lower bounds only.
   NumPy is not listed because the package never imports it directly.

**Interpretation.** The numbers are irrelevant; the result that matters is the pair of
facts *"same output as before"* and *"independent of the working directory"*.

**Reproducibility.** Deterministic: committed data, no randomness. Verified with the
versions in `docs/maintainers/tested-versions.md`.

---

## 4. Guided lab

Workspace: [`lab/`](../../labs/w1d3_package_structure/lab/README.md). Starter material: `messy_analysis.py` (the script),
`stubs/` (module contracts), `REFACTOR_LOG.md` (template), `compare_to_golden.py`.

**Result to produce:** an installed `pumpwatch` command that reproduces the script's
metrics from any directory, with the scaler leakage removed as a separate logged change.

<img src="https://raw.githubusercontent.com/berdakh/mlengineering/main/docs/assets/diagrams/mmd-03-03.svg" alt="Diagram 3 of notebook 03" style="max-width:100%">

*The lab in one picture: preserve behaviour first and prove it, then change behaviour once, on purpose.*

### Task 0 — Interpreter and environment (10 min)

```bash
python3 --version          # need >= 3.11 (tomllib and the package's requires-python)
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install numpy pandas scikit-learn pytest
```

If the repository already has a `.venv` from the course setup, reuse it. On Ubuntu
22.04 and its derivatives the system interpreter is 3.10: see H2.1 § "Getting a newer interpreter"
(`uv python install 3.12` then `uv venv --python 3.12 .venv` is the shortest route;
if `python3 -m venv` complains about `ensurepip`, install `python3-venv` with apt).

**Checkpoint.** `python -c "import sys; print(sys.prefix != sys.base_prefix, sys.executable)"`
prints `True` and a path inside `.venv`.

### Task 1 — Characterize before touching anything (5 min)

```bash
python labs/w1d3_package_structure/lab/messy_analysis.py | tee labs/w1d3_package_structure/lab/golden.txt
```

Expected: `(300, 10)`, test units `['P01', 'P06', 'P09']`, `auc ≈ 0.9992`, `ap ≈ 0.9971`,
`acc ≈ 0.9667`, `f1 ≈ 0.9091`. Run it twice and `diff` the outputs: is anything
non-deterministic? Record the answer in `REFACTOR_LOG.md`.

Then **read** the script and list every problem you see in the log's issue table —
structural, statistical, anything. Do not fix any of them yet. (There are at least eight.)

### Task 2 — Skeleton and editable install (10 min)

Create `lab/pyproject.toml` and `lab/src/pumpwatch/__init__.py`.
Required in `pyproject.toml`: `[build-system]` (setuptools), `[project]` with
`name = "pumpwatch"`, `version = "0.1.0"`, `requires-python`, the runtime `dependencies`
the code imports, and package discovery under `src`
(`[tool.setuptools.packages.find] where = ["src"]`). Use the worked example as a model —
type it, do not copy the file.

```bash
python -m pip install -e labs/w1d3_package_structure/lab
cd /tmp && python -c "import pumpwatch; print(pumpwatch.__file__, pumpwatch.__version__)"; cd -
```

**Checkpoint.** Prints a path ending in `lab/src/pumpwatch/__init__.py` and `0.1.0`,
from `/tmp`.

### Task 3 — Features (15 min)

Copy `stubs/features.py` to `src/pumpwatch/features.py` and fill it in from the script.
The two band-power blocks are copy-paste twins: write `band_power` once. Keep the
arithmetic *identical* — same detrending, same normalisation, same `[low, high)` band
edges — even where you would choose differently. Improvements come later and separately.

**Checkpoint** (paste into `python`, from the repo root):

```python
import contextlib, io, runpy
import numpy as np
from pumpwatch.features import build_feature_table
import pandas as pd

with contextlib.redirect_stdout(io.StringIO()):
    script = runpy.run_path("labs/w1d3_package_structure/lab/messy_analysis.py")
old = script["feat"].sort_values(["unit_id", "window_id"]).reset_index(drop=True)
new = build_feature_table(pd.read_csv("data/synthetic/pump_vibration.csv"), fs=256)
new = new.sort_values(["unit_id", "window_id"]).reset_index(drop=True)
cols = ["rms", "line_length", "p2p", "bp_low", "bp_high", "ratio", "temp", "label"]
print(np.abs(old[cols].to_numpy(float) - new[cols].to_numpy(float)).max())
```

Expected: `0.0`, or at most ~1e-15. Anything like 1e-3 is a real difference — find it.

### Task 4 — I/O, split, model: a *faithful* port (10 min)

Copy and complete `stubs/io.py`, `stubs/split.py`, `stubs/model.py`. Port what the script
does, **including the scaler fitted on all rows**. `make_model(C)` returns the
classifier; `train_and_evaluate` returns a dict of metrics.

Why port a known flaw? Because if you fix it during the move and a number changes, you
cannot tell whether the fix or a porting mistake caused it.

### Task 5 — CLI and entry point (10 min)

Copy and complete `stubs/cli.py`; add to `pyproject.toml`:

```toml
[project.scripts]
pumpwatch = "pumpwatch.cli:main"
```

Re-run `python -m pip install -e labs/w1d3_package_structure/lab` (new entry point ⇒
metadata changed ⇒ reinstall). Then:

```bash
pumpwatch train --data data/synthetic/pump_vibration.csv > /tmp/metrics.json
python labs/w1d3_package_structure/lab/compare_to_golden.py \
    labs/w1d3_package_structure/lab/golden.txt /tmp/metrics.json
```

**Checkpoint.** Four `ok` lines. Log it as a behaviour-preserving step.

### Task 6 — Prove independence from the working directory (5 min)

```bash
REPO=$(pwd); cd /tmp
pumpwatch train --data "$REPO/data/synthetic/pump_vibration.csv" | head -3
pumpwatch train --data nope.csv; echo "exit code: $?"
cd "$REPO"
```

**Checkpoint.** Metrics from `/tmp`; for the missing file a one-line `error: …` on stderr,
no traceback, exit code 2.

### Task 7 — One deliberate behaviour change: fix the leak (10 min)

Change `make_model` so that it returns a single estimator that scales *and* classifies
(`sklearn.pipeline.make_pipeline`), and remove the scaling from `train_and_evaluate`.
Before running: write down in the log whether you expect the metrics to go up, down, or
stay, and why. Then run and record before/after.

You will find the four printed metrics **unchanged to every digit**. Explain that in the
log's "Interpretation" column before reading on.

<details><summary>Discussion (open after writing your interpretation)</summary>

The leak is real: standardisation used the mean and variance of rows from the held-out
units. And the fitted model *did* change — in the reference run the scaler fitted on
training units only has scales up to ~13 % different and means shifted by up to ~0.17 SD
compared with the all-rows scaler, so the coefficients and the individual probabilities
are different. What did not change is anything the script *prints*: (a) AUC and AP depend
only on the ranking of test scores, which was preserved, and (b) accuracy and F1 depend
only on which side of 0.5 each probability falls, and the closest test probability is
about 0.10 away from the threshold. Four summary numbers are a coarse instrument. On a harder
problem, with fewer groups, heavier-tailed features, or a preprocessing step with more
capacity than a mean and a variance (feature selection, PCA, target encoding, imputation
models), the same structural mistake inflates results materially.

So the justification for the change is **by construction**: after it, test rows *cannot*
influence fitted preprocessing, on any dataset. "The number did not move" is evidence
that the leak was small here; it is not evidence that there was no leak, and it would be
wrong to conclude the fix was unnecessary. Week 3 returns to this with data where the
number does move.
</details>

### Task 8 — Run the checks, finish the log (5 min)

In [ ]:
!python -m pytest labs/w1d3_package_structure/checks -q

Expected: `13 passed, 1 skipped`. Optional, needs network and about a minute — a real
install into a fresh venv: `WB_RUN_SLOW=1 python -m pytest labs/w1d3_package_structure/checks -q`.

### Debugging guide

| Symptom | Likely cause | What to look at |
|---|---|---|
| `ModuleNotFoundError: No module named 'pumpwatch'` | not installed in *this* interpreter, or discovery found no packages | `python -m pip show -f pumpwatch`; `which python`; is `where = ["src"]` set and does `src/pumpwatch/__init__.py` exist? |
| `import pumpwatch` works in `lab/` but not in `/tmp` | you are importing from the working directory, not an install (flat layout or stray `pumpwatch/` folder) | `python -c "import pumpwatch; print(pumpwatch.__file__)"` |
| `pumpwatch: command not found` | entry point added after the install; or venv not active | reinstall with `-e`; `ls .venv/bin/pumpwatch` |
| `ImportError: cannot import name 'main'` when running `pumpwatch` | `[project.scripts]` names the wrong module or function | the string is `"package.module:function"` |
| `pip install -e` fails with "Multiple top-level packages discovered" | setuptools auto-discovery in a flat layout saw several directories | use `src/` and `where = ["src"]` |
| `externally-managed-environment` | you ran pip against the system Python | create/activate the venv; use `python -m pip` |
| Metrics differ in the 3rd–4th decimal | band edges (`<` vs `<=`), detrending, NaN handling, or row order changed | Task 3 checkpoint, feature by feature |
| Different test units | `GroupShuffleSplit` got different groups/order or a different seed | pass `groups=features["unit_id"]`, `random_state=seed` |
| A different copy of `pumpwatch` is imported | another lab's package was installed with the same name later | `pip show -f pumpwatch`; reinstall the one you want (one distribution name, one install) |

---

In [ ]:
# Run the automated checks on your guided lab.
# A fresh starter is expected to FAIL; passing means guided practice, not mastery.
from mlcourse.checks import check

check("w1d3")

## 5. Independent challenge

[`challenge/README.md`](../../labs/w1d3_package_structure/challenge/README.md) — extend the package to a two-stage
pipeline (`pumpwatch features` → `pumpwatch train --features`) that also runs as
`python -m pumpwatch`. Acceptance criteria are listed there; no solution is given.
Checks: `python -m pytest labs/w1d3_package_structure/checks_challenge -q`.

Work in `challenge/` (start by copying your finished `lab/` package into it).

---


### The challenge brief


**Time:** about 45 minutes. **Assistance:** none to begin with; hints are in
`../hints/` and cost you the "independent" flag from hint 3 onward.

#### Situation

Feature extraction is the slow part of real pipelines (on the project dataset it will
be minutes, not milliseconds), and several people want the feature table without
training anything. A teammate also runs tools as `python -m <package>` because their
scheduler does not put a virtual environment's `bin/` on `PATH`.

Extend `pumpwatch` so that features can be computed once, saved, and reused.

#### Setup

Copy your finished lab package into this directory and install *this* copy:

```bash
cp -r labs/w1d3_package_structure/lab/pyproject.toml labs/w1d3_package_structure/lab/src \
      labs/w1d3_package_structure/challenge/
python -m pip install -e labs/w1d3_package_structure/challenge
python -c "import pumpwatch; print(pumpwatch.__file__)"     # must now point into challenge/
```

(One distribution name can be installed once per environment; the second install
replaces the first. Re-install `lab/` later if you want it back.)

#### Acceptance criteria

1. `pumpwatch features --data <raw.csv> --out <features.csv>` writes a CSV with exactly
   one row per `(unit_id, window_id)` and at least the columns
   `unit_id, window_id, rms, line_length, p2p, bp_low, bp_high, ratio, temp, label`.
   Missing parent directories of `--out` are created. Exit code 0. Nothing but data in the file
   (no index column). Standard output is either empty or not relied upon.
2. `pumpwatch train --features <features.csv>` trains from the saved table and prints the
   same JSON metrics as `pumpwatch train --data <raw.csv>` (to 1e-6).
3. `train` takes **exactly one** of `--data` / `--features`: neither, or both, is a usage
   error with a non-zero exit code.
4. `python -m pumpwatch <args>` behaves identically to `pumpwatch <args>`.
5. A missing input file — for either subcommand and either option — gives a one-line
   message on stderr, **exit code 2**, no traceback, and leaves no output file behind.
6. `pumpwatch train --features <a file that is not a feature table>` (for example the raw
   CSV) fails the same clean way: exit code 2, readable message, no traceback.
7. The version is `0.2.0` — in `pyproject.toml`, in `pumpwatch.__version__`, and in
   `pumpwatch --version`.
8. All of the above hold when the command is run from a directory outside the repository.

#### Check yourself

In [ ]:
!python -m pytest labs/w1d3_package_structure/checks_challenge -q

#### What to hand in for feedback

The `challenge/` directory, plus three or four sentences in `challenge/NOTES.md`:
where you put the new read/write functions and why there; what you had to re-run after
editing `pyproject.toml`; and one thing about the saved-features design you would
question before using it on a real project (think: what does the file *not* record?).

#### Out of scope

Configuration files and logging (w1d4), unit tests (w1d5), Parquet or other formats.

In [ ]:
# Run the automated checks on your independent challenge.
# A fresh starter is expected to FAIL; passing means guided practice, not mastery.
from mlcourse.checks import check

check("w1d3", challenge=True)

## 6. Progressive hints

Open only as far as you need, and note in `progress` how far you went.

1. [Conceptual hint](../../labs/w1d3_package_structure/hints/hint1_concept.md)
2. [Suggested approach](../../labs/w1d3_package_structure/hints/hint2_approach.md)
3. [Partial implementation](../../labs/w1d3_package_structure/hints/hint3_partial.md)
4. Full reference solution: `solutions/labs/w1d3_package_structure/` (lab and challenge).
   Opening it before your own attempt passes means the unit counts as *guided*, not *independent*.

---


Open one hint at a time, only after a real attempt.

<details>
<summary><b>Hint 1 concept</b></summary>


- A console script and `python -m package` are two *different* launch mechanisms.
  The first is generated from `[project.scripts]`. For the second, Python looks for a
  specific module **inside the package**. `python -m pumpwatch.cli` working tells you
  nothing about `python -m pumpwatch`.
- Think about where each responsibility belongs before writing code. You already have a
  module that owns "turn a path into a validated DataFrame". Writing and re-reading the
  feature table is the same kind of job. The CLI should stay a thin layer that parses
  arguments, calls functions, and translates *expected* failures into exit codes.
- "Exactly one of two options" is a standard argument-parsing situation; `argparse` has
  direct support for it. You do not need to hand-write the check.
- Every way of getting data in is a boundary, and every boundary needs the same care.
  When you add a second loader, ask what the first loader checks that the second should too.
- Criterion 5 says "for either subcommand". If your error handling wraps only the code
  path that existed before, the new path is unprotected.
- Changing the version is a metadata change. Remember what an editable install does and
  does not pick up automatically.

Lab hints: if you are stuck on the lab rather than the challenge, the lesson's
debugging table (section 4) is the place to look first.

</details>

<details>
<summary><b>Hint 2 approach</b></summary>


Work in this order, running the command by hand after each step.

1. **`__main__.py`.** Add `src/pumpwatch/__main__.py`. It needs three lines: import
   `main` from the CLI module and exit with its return value. Check with
   `python -m pumpwatch --version`.
2. **I/O functions** in `io.py`:
   - `save_features(features, path) -> Path`: create `path.parent` (`parents=True,
     exist_ok=True`), then `to_csv(..., index=False)`.
   - `load_features(path) -> DataFrame`: same shape as `load_windows` — existence check,
     read, required-column check raising `ValueError`.
3. **Parser.** Add a `features` subparser (`--data`, `--out`, `--fs`). In the `train`
   subparser replace the required `--data` with a *mutually exclusive group* that is
   itself required, holding `--data` and `--features`.
4. **Dispatch.** Put the work in one function that takes the parsed arguments and
   returns an exit code, and wrap the *single call to it* in the `try/except` that maps
   `FileNotFoundError` / `ValueError` to exit code 2. That way a third subcommand cannot
   forget the handling.
5. **Order of operations for `features`.** Load and compute first, write last. If the
   input is missing you fail before anything is created on disk (criterion 5).
6. **Version.** Change it in `pyproject.toml` and `__init__.py`, then reinstall with
   `python -m pip install -e …/challenge` and look at `pumpwatch --version`.
7. **Round-trip sanity check.** `pumpwatch features … --out /tmp/x/y/f.csv` then
   `pumpwatch train --features /tmp/x/y/f.csv` and compare with `--data`. If the metrics
   differ slightly, look at what `to_csv` did to your floats or your column order
   (`FEATURE_COLUMNS` should select by *name*, never by position).

</details>

<details>
<summary><b>Hint 3 partial</b></summary>


Using this hint means the unit is recorded as guided, not independent.

`src/pumpwatch/__main__.py`:

```python
import sys

from pumpwatch.cli import main

sys.exit(main())
```

Parser fragment:

```python
train = sub.add_parser("train", help="train and print test metrics as JSON")
source = train.add_mutually_exclusive_group(required=True)
source.add_argument("--data", help="raw window CSV")
source.add_argument("--features", help="feature CSV written by `pumpwatch features`")
```

`main` skeleton — the body of `_run` and the two I/O functions are yours:

```python
def _run(args) -> int:
    if args.command == "features":
        ...  # load raw -> build table -> save; return 0
    ...      # train: pick the source, then train_and_evaluate; print JSON; return 0


def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    try:
        return _run(args)
    except (FileNotFoundError, ValueError) as exc:
        print(f"error: {exc}", file=sys.stderr)
        return 2
```

Loader skeleton:

```python
def load_features(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(...)
    features = pd.read_csv(path)
    missing = [c for c in FEATURE_TABLE_COLUMNS if c not in features.columns]
    ...  # what should happen here, and which exception type keeps main() simple?
```

Still stuck? The full solution is in
`solutions/labs/w1d3_package_structure/challenge/`. Read it, close it, and redo the
challenge from an empty directory tomorrow.

</details>

## 7. Assessment

`assessment.md` (section 7 of this notebook): six short reasoning questions (one is a design-choice
explanation), the rubric for open answers, and one timed interview rep. Write your
answers in `lab/ASSESSMENT_ANSWERS.md` *before* opening
`solutions/labs/w1d3_package_structure/assessment_answers.md`, which also lists the
common incorrect answers and why they fail.

Automated checks: `checks/` (lab) and `checks_challenge/` (challenge). They test the
packaging contract and behaviour; they do not test whether you could do it again unaided.

---


### Assessment questions


Write your answers in `lab/ASSESSMENT_ANSWERS.md` **before** opening
`solutions/labs/w1d3_package_structure/assessment_answers.md`. Two to five sentences
each; no code needed unless asked. Budget: 10 minutes for Q1–Q6, 15 for the interview rep.

#### Reasoning questions

**Q1.** You are in a project root that uses a *flat* layout. `python -m pytest` passes.
You then run `pip uninstall pumpwatch` and `python -m pytest` again, and it still
passes. Explain precisely why, and say what this implies about what the tests were
exercising all along.

**Q2.** After adding `[project.scripts] pumpwatch = "pumpwatch.cli:main"` to an already
editable-installed project, typing `pumpwatch` gives "command not found", yet
`python -m pumpwatch.cli` works. Why, and what is the fix? Name one other kind of
`pyproject.toml` edit with the same consequence.

**Q3.** A colleague proposes replacing the `dependencies` list in `pyproject.toml` with
the 140 exact pins from their `pip freeze`. Give one situation where exact pins are the
right tool and one concrete way this proposal causes harm. Where *should* the pins live?

**Q4 (design choice).** In the lab you were told to port the scaler-before-split
behaviour faithfully and to fix it only afterwards as a separate step. A reviewer says:
"That is wasteful — you knew it was a bug, you should have fixed it while moving the
code." Defend the two-step order. Then state one circumstance in which the reviewer
would be right.

**Q5.** After the leak fix the four printed metrics were identical to every digit.
Three people react:
(a) "So there was no leakage." (b) "So the fix was unnecessary; revert it to keep the
diff small." (c) "So this check cannot detect that leak; keep the fix and note the
limitation." Which is correct, and what specifically is wrong with the other two?

**Q6.** `main()` returns an integer and the module ends with `sys.exit(main())`, rather
than calling `sys.exit(2)` deep inside the function that reads the file. Give two
practical benefits of this arrangement.

#### Rubric for open answers (Q4 and Q5; apply the same scale to the reflection prompt)

| Level | Description |
|---|---|
| **3 — Sound** | States the mechanism correctly, ties it to *this* lab's evidence (the unchanged metrics, the golden comparison), and names a limitation or a condition under which the conclusion would change. |
| **2 — Mostly right** | Correct conclusion and mechanism, but generic: no reference to the evidence, or no boundary condition. |
| **1 — Right word, wrong reason** | Reaches the accepted conclusion using a reason that does not hold (e.g. "always refactor first because that is best practice"; "leakage always inflates metrics"). |
| **0 — Incorrect or absent** | Wrong conclusion, or an answer that restates the question. |

Self-score honestly, then compare with the model answers. A 1 is more useful to record
than an optimistic 2: it tells the review day what to revisit.

#### Interview rep (15 minutes, timed, no assistance, no scikit-learn)

> Implement
>
> ```python
> def group_split(groups: list[str], test_fraction: float, seed: int) -> tuple[list[int], list[int]]:
>     """Return (train_indices, test_indices) into `groups`."""
> ```
>
> such that: no group label appears on both sides; the *number of test groups* is
> `round(test_fraction * n_groups)` but never 0 and never all of them; the result is
> identical for the same `seed` regardless of the order in which rows arrive being
> permuted **together with** their indices (i.e. which *groups* are test depends only on
> the set of group labels and the seed); indices come back sorted ascending.
> Raise `ValueError` if there are fewer than two distinct groups or if
> `test_fraction` is not strictly between 0 and 1.

Say your approach out loud (or write two lines) before coding. When the timer ends,
stop, then test your function with the snippet in the answers file and score yourself
with the rubric there. Assistance is reduced on purpose: there are no hints for this one.

## 8. Reflection and review

**You should now be able to, without notes:**

- start a new `src`-layout project and get `pip install -e .` plus a console command working;
- diagnose "module not found" by reasoning about `sys.path` rather than by trial and error;
- state the difference between declared dependencies and a lock, and who needs which;
- carry out a refactor as a sequence of verified behaviour-preserving steps, with
  behaviour changes isolated and logged.

**Reflection prompt (5 min, write it in the log).** Pick one repository from your own
research. If a new student cloned it today, which of this lesson's failure modes would
they hit first, and what is the smallest change that would remove it?

**Retrieval practice — answer on the Week 1 review day, from memory:**
> You run `python -m pytest` in a project root and all tests pass. A colleague installs
> the wheel you built from the same commit and gets `ModuleNotFoundError` for one of your
> subpackages. Give the mechanism that lets both be true, and two independent changes
> that would have exposed the problem on your machine.

**Mark progress** (reading / guided / independent / review-needed are separate flags):

```bash
python wb.py mark w1d3 reading
python wb.py mark w1d3 guided        # after the lab checks pass
python wb.py mark w1d3 independent   # only after the challenge, unaided
```

**Next: w1d4 — Configuration, logging and reproducible runs.** The literals you noted in
the issue table (`fs`, band edges, `C`, seed, test fraction) are still scattered through
code and CLI defaults, and a run leaves no trace of which values produced which numbers.
w1d4 gives them one typed home and makes every run write down what it did.

---

*Content status: see `STATUS.md`. Sources and what was verified: handbook H2.1, "Sources and verification status".*

## Record your progress

Tick only what is true. Passing the supplied checks is *guided practice*; mark *independent* only for work done without hints or solutions.

In [ ]:
from mlcourse.progress import mark, note

# mark("w1d3", "reading")
# mark("w1d3", "guided")
# mark("w1d3", "independent")
# note("w1d3", "what tripped me up today")